In [1]:
from functools import partial
import pandas as pd
from datasets import load_dataset
data_path = ["data_raw/train.txt","data_raw/valid.txt","data_raw/test.txt"]
data_output_path = ["data_raw/train.jsonl","data_raw/valid.jsonl","data_raw/test.jsonl"]
from modelscope import GPT2Tokenizer, GPT2LMHeadModel
hf_model_path = 'Fengshenbang/Wenzhong-GPT2-110M-chinese-v2'
tokenizer = GPT2Tokenizer.from_pretrained(hf_model_path)
# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(batch,tokenizer):
    return tokenizer(batch['context'])

dataset = load_dataset('json', data_files={
    'train': data_output_path[0],
    'valid': data_output_path[1],
    'test': data_output_path[2]
})
# 查看结构
print(dataset['train'][0])
# 输出：{'idx': 0, 'context': '如果 我 无聊 时 ...', 'label': 1}

map_kwargs = {
    'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
    'batch_size': 512,  # 每批 512 个样本
    'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
}

# 在 map 时绑定 tokenizer
tokenize_with_tokenizer = partial(tokenize, tokenizer=tokenizer)

# 对训练集和验证集应用 map
tokenized_dataset_train = dataset["train"].map(tokenize_with_tokenizer, **map_kwargs)
tokenized_dataset_val = dataset["valid"].map(tokenize_with_tokenizer, **map_kwargs)

print(tokenized_dataset_train[0])
print(tokenized_dataset_val[0])



D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.regi

{'idx': 0, 'context': '死囚爱刽子手女贼爱衙役我们爱你们难道还有别的选择没想到胡军除了蓝宇还有东宫西宫我个去阿兰这样真他nia恶心爱个P分明只是欲', 'label': 1}


Map: 100%|██████████| 5629/5629 [00:09<00:00, 585.83 examples/s]

{'input_ids': [29826, 119, 32368, 248, 163, 230, 109, 26344, 121, 36310, 33699, 233, 42637, 164, 112, 120, 163, 230, 109, 26193, 247, 37605, 117, 22755, 239, 20015, 105, 163, 230, 109, 19526, 254, 20015, 105, 49694, 122, 34402, 241, 32573, 246, 17312, 231, 26344, 104, 21410, 34460, 231, 162, 233, 102, 162, 110, 94, 46349, 111, 26344, 108, 47797, 94, 37863, 249, 165, 247, 97, 12859, 228, 164, 241, 251, 22522, 229, 32573, 246, 17312, 231, 10310, 250, 22522, 104, 164, 98, 123, 22522, 104, 22755, 239, 10310, 103, 43889, 119, 165, 246, 123, 17739, 108, 32573, 247, 43718, 115, 40367, 253, 20015, 244, 18142, 162, 223, 114, 33232, 225, 163, 230, 109, 10310, 103, 47, 26344, 228, 23626, 236, 20998, 103, 42468, 162, 105, 110], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [2]:
for i, seq in enumerate(tokenized_dataset_train[5:10]['input_ids']):
    print(f'{i+1}: {tokenizer.decode(seq)}')

#清理数据长度国小的
def filter_short_samples(batch, min_length=10):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['input_ids'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)
# 对训练集和验证集应用过滤函数
filtered_dataset_train = tokenized_dataset_train.filter(filter_short_samples)
filtered_dataset_val = tokenized_dataset_val.filter(filter_short_samples)
print(f"训练集过滤前样本数量: {len(tokenized_dataset_train)}")
print(f"训练集过滤后样本数量: {len(filtered_dataset_train)}")
print(f"验证集过滤前样本数量: {len(tokenized_dataset_val)}")
print(f"验证集过滤后样本数量: {len(filtered_dataset_val)}")

#放入torch
import torch
filtered_dataset_train.set_format(type='torch')
filtered_dataset_val.set_format(type='torch')
print(filtered_dataset_train[0])
print(filtered_dataset_val[0])
# 计算验证集中 input_ids 的最大长度
max_len = max(len(seq) for seq in filtered_dataset_val['input_ids'])
print(f"验证集最大序列长度: {max_len}")


1: 这种忒著名历史事件不能严谨尊重历史一点除了预计撒自刎乌江四面楚歌细节没有太还原实属主要角色刘邦项羽性格也塑造失败陈小春饰演樊哙算性格鲜明黎明上不起不想评论直接败坏影片印象分
2: 导演你猪脑子几场著名战役被你拍动不动有人跳出来说宋江哥哥我妙计一条尼玛个妙计本来好好妙计被你给拍成屎不知道以为梁山一帮子爷们儿脑残弱智这样计也敢称妙计怎么着梁山也人才辈出特技花
3: 弱智剧情没关系可以忍垃圾剪辑没关系可以忍东倒西歪三观没关系可以忍坑爹口音没关系可以忍陈坤油腻腻油头没关系还是可以忍但是志玲姐姐没有胸真的不能忍
4: 第一次看很小没见过这么露骨片子当黄片看后来来英国之后又看一遍没什么意思萨朗斯通后背挺多肥肉现在越开她不顺眼大妈到底谁
5: 我说赵氏孤儿怎么那么拧巴想来想去不喜欢陈凯歌通过扭曲青少年心理制造人间悲剧叙事模式先是拍一个馒头引发血案这次又拍这么一个少年一个爹养他一个爹教他一个爹他他养父唆使下为了给他生父报仇杀他教父这是哪门子仁义礼智信


Filter: 100%|██████████| 5629/5629 [00:00<00:00, 16981.27 examples/s]


训练集过滤前样本数量: 19998
训练集过滤后样本数量: 7623
验证集过滤前样本数量: 5629
验证集过滤后样本数量: 2093
{'input_ids': tensor([29826,   119, 32368,   248,   163,   230,   109, 26344,   121, 36310,
        33699,   233, 42637,   164,   112,   120,   163,   230,   109, 26193,
          247, 37605,   117, 22755,   239, 20015,   105,   163,   230,   109,
        19526,   254, 20015,   105, 49694,   122, 34402,   241, 32573,   246,
        17312,   231, 26344,   104, 21410, 34460,   231,   162,   233,   102,
          162,   110,    94, 46349,   111, 26344,   108, 47797,    94, 37863,
          249,   165,   247,    97, 12859,   228,   164,   241,   251, 22522,
          229, 32573,   246, 17312,   231, 10310,   250, 22522,   104,   164,
           98,   123, 22522,   104, 22755,   239, 10310,   103, 43889,   119,
          165,   246,   123, 17739,   108, 32573,   247, 43718,   115, 40367,
          253, 20015,   244, 18142,   162,   223,   114, 33232,   225,   163,
          230,   109, 10310,   103,    47, 26344,   228, 23

In [3]:
#填充
print(tokenizer.pad_token)
print(tokenizer.eos_token)
#将填充标记设置为 EOS 标记
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling
# mlm=False，将数据整理成“因果语言建模”需要的数据格式
# “因果语言建模”就是“预测下一个token”类型的任务，也就是gpt风格的自回归模型
# 如果mlm=True，那么数据整理成bert风格的任务所需的数据格式
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False) # labels

dataloader_params = {
    'batch_size': 16, #根据需要调整批量大小
    'collate_fn': data_collator
}

train_dataloader = DataLoader(filtered_dataset_train, **dataloader_params)
val_dataloader = DataLoader(filtered_dataset_val, **dataloader_params)

print(len(train_dataloader))

batch = next(iter(train_dataloader))
print(batch.keys())
print(batch['input_ids'].shape)
print(batch['input_ids'][0])
print(batch['labels'][0])
print(batch['attention_mask'][0])

None
<|endoftext|>
477
dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([16, 148])
tensor([29826,   119, 32368,   248,   163,   230,   109, 26344,   121, 36310,
        33699,   233, 42637,   164,   112,   120,   163,   230,   109, 26193,
          247, 37605,   117, 22755,   239, 20015,   105,   163,   230,   109,
        19526,   254, 20015,   105, 49694,   122, 34402,   241, 32573,   246,
        17312,   231, 26344,   104, 21410, 34460,   231,   162,   233,   102,
          162,   110,    94, 46349,   111, 26344,   108, 47797,    94, 37863,
          249,   165,   247,    97, 12859,   228,   164,   241,   251, 22522,
          229, 32573,   246, 17312,   231, 10310,   250, 22522,   104,   164,
           98,   123, 22522,   104, 22755,   239, 10310,   103, 43889,   119,
          165,   246,   123, 17739,   108, 32573,   247, 43718,   115, 40367,
          253, 20015,   244, 18142,   162,   223,   114, 33232,   225,   163,
          230,   109, 10310,   103,    47, 2

In [4]:
#加载模型监督微调
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GPT2LMHeadModel.from_pretrained(hf_model_path).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
#训练一个epoch
num_epochs = 1

In [5]:
# 训练循环
#先评估一下初始模型在验证集上的表现
def validate(epoch):
    model.eval()
    total_loss = 0.0
    for i, batch in enumerate(val_dataloader):
        batch = batch.to(device)
        with torch.no_grad():
            outputs = model(**batch)
            loss = outputs.loss # 损失
            total_loss += loss.item()
    print(f'val_loss at {epoch} epoch:', total_loss / len(val_dataloader))
validate(1)
#正式训练
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for i, batch in enumerate(train_dataloader):
        batch = batch.to(device)
        outputs = model(**batch)
        loss = outputs.loss # 损失
        total_loss += loss.item()
        optimizer.zero_grad() # 梯度清零
        loss.backward() # 反向传播
        optimizer.step() # 更新参数
        print(f'train_loss at {i} epoch:', loss.item())
    print("train_loss at {epoch} epoch:", total_loss / len(train_dataloader))
    validate(epoch+1)

val_loss at 1 epoch: 2.090357731316836
train_loss at 0 epoch: 0.13364005088806152
train_loss at 1 epoch: 0.14161480963230133
train_loss at 2 epoch: 0.1426665335893631
train_loss at 3 epoch: 0.11996692419052124
train_loss at 4 epoch: 0.12518712878227234
train_loss at 5 epoch: 0.1299746185541153
train_loss at 6 epoch: 0.13027577102184296
train_loss at 7 epoch: 0.12479768693447113
train_loss at 8 epoch: 0.1323414295911789
train_loss at 9 epoch: 0.12110283970832825
train_loss at 10 epoch: 0.12469197809696198
train_loss at 11 epoch: 0.13390477001667023
train_loss at 12 epoch: 0.12656265497207642
train_loss at 13 epoch: 0.1344703733921051
train_loss at 14 epoch: 0.1307816505432129
train_loss at 15 epoch: 0.12283370643854141
train_loss at 16 epoch: 0.1266501098871231
train_loss at 17 epoch: 0.1166597381234169
train_loss at 18 epoch: 0.12097247689962387
train_loss at 19 epoch: 0.12450946867465973
train_loss at 20 epoch: 0.13036233186721802
train_loss at 21 epoch: 0.12509198486804962
train_loss

In [6]:
#保存微调后的模型
model.save_pretrained('models/fine_tuned_gpt2')
tokenizer.save_pretrained('models/fine_tuned_gpt2')

('models/fine_tuned_gpt2\\tokenizer_config.json',
 'models/fine_tuned_gpt2\\special_tokens_map.json',
 'models/fine_tuned_gpt2\\vocab.json',
 'models/fine_tuned_gpt2\\merges.txt',
 'models/fine_tuned_gpt2\\added_tokens.json')